# 02 HR Feature Engineering

Professional People Analytics notebook for Employee Attrition Prediction & Workforce Intelligence.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "data" / "raw" / "WA_Fn-UseC_-HR-Employee-Attrition.csv").exists():
            return candidate
    raise RuntimeError("Project root not found. Open the notebook from the repository or one of its subfolders.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('D:/Project/Data Science/employee_attrition_prediction_&_workforce_intelligence')

## Objective

Create HR analytics features that are transparent, reproducible, and safe for modeling. Identifier and constant fields are excluded from the feature matrix while retained where needed for reporting.

In [2]:
import pandas as pd

from src.config import CONSTANT_COLUMNS, IDENTIFIER_COLUMN, TARGET_COLUMN, TARGET_FLAG
from src.data import load_raw_data
from src.features import add_hr_features, build_modeling_dataset

raw = load_raw_data()
featured = add_hr_features(raw)
X, y, featured_for_modeling = build_modeling_dataset(raw)

X.shape, y.mean(), featured.shape

((1470, 55), np.float64(0.16122448979591836), (1470, 61))

In [3]:
leakage_columns = {TARGET_COLUMN, TARGET_FLAG, IDENTIFIER_COLUMN, *CONSTANT_COLUMNS}
sorted(leakage_columns.intersection(X.columns))

[]

## Engineered Feature Preview

In [4]:
engineered_columns = [
    "age_band",
    "income_band",
    "distance_band",
    "tenure_band",
    "overtime_flag",
    "frequent_travel_flag",
    "early_tenure_flag",
    "career_stagnation_flag",
    "satisfaction_score",
    "engagement_score",
    "career_growth_score",
    "compensation_to_job_level_ratio",
    "promotion_gap_ratio",
    "attrition_risk_segment",
]
featured[["EmployeeNumber", "Attrition", *engineered_columns]].head(12)

,EmployeeNumber,Attrition,age_band,income_band,distance_band,tenure_band,overtime_flag,frequent_travel_flag,early_tenure_flag,career_stagnation_flag,satisfaction_score,engagement_score,career_growth_score,compensation_to_job_level_ratio,promotion_gap_ratio,attrition_risk_segment
0,1,Yes,36-45,Developing income,Near,4-7 years,1,0,0,0,2.00,2.50,0.475057,2996.500000,0.000000,Satisfaction risk
1,2,No,46-55,Developing income,Moderate,8-15 years,0,1,0,0,3.00,2.75,0.666213,2565.000000,0.090909,Satisfaction risk
2,4,Yes,36-45,Entry income,Near,0-1 years,1,0,1,0,3.00,2.75,0.526077,2090.000000,0.000000,Mobility and reward risk
3,5,No,26-35,Entry income,Near,8-15 years,1,1,0,0,3.25,3.25,0.164172,2909.000000,0.333333,Mobility and reward risk
4,7,No,26-35,Developing income,Near,2-3 years,0,0,1,0,2.50,2.50,0.234354,3468.000000,0.666667,Satisfaction risk
5,8,No,26-35,Developing income,Near,4-7 years,0,1,0,0,3.25,3.50,0.256576,3068.000000,0.375000,Mobility and reward risk
6,10,No,56-60,Entry income,Near,0-1 years,1,0,1,0,1.75,2.25,0.617460,2670.000000,0.000000,Early-tenure workload risk
7,11,No,26-35,Entry income,Very far,0-1 years,0,0,1,0,3.00,3.00,0.640930,2693.000000,0.000000,Baseline workforce
8,12,No,36-45,Mid income,Very far,8-15 years,0,1,0,0,3.00,2.75,0.730159,3175.333333,0.100000,Mobility and reward risk
9,13,No,36-45,Developing income,Very far,4-7 years,0,0,0,1,2.50,2.75,0.332880,2618.500000,0.875000,Career progression risk


## Segment-Level Validation

In [5]:
(
    featured.groupby("attrition_risk_segment")
    .agg(
        employees=("attrition_flag", "size"),
        attrition_rate=("attrition_flag", "mean"),
        avg_satisfaction=("satisfaction_score", "mean"),
        overtime_share=("overtime_flag", "mean"),
    )
    .sort_values("attrition_rate", ascending=False)
)

,employees,attrition_rate,avg_satisfaction,overtime_share
attrition_risk_segment,,,,
Early-tenure workload risk,64,0.562500,2.574219,1.000000
Satisfaction risk,697,0.173601,2.483859,0.213773
Mobility and reward risk,226,0.154867,3.180310,0.318584
Career progression risk,285,0.122807,2.746491,0.259649
Baseline workforce,198,0.050505,3.116162,0.287879


In [6]:
feature_audit = pd.DataFrame(
    {
        "feature_count": [X.shape[1]],
        "numeric_features": [len(X.select_dtypes(include="number").columns)],
        "categorical_features": [len(X.select_dtypes(include=["object", "category"]).columns)],
        "missing_values": [int(X.isna().sum().sum())],
    }
)
feature_audit

,feature_count,numeric_features,categorical_features,missing_values
0,55,41,14,0
